In [1]:
import torch
from datasets import load_dataset
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

# Load dataset
dataset = load_dataset("timm/eurosat-rgb")
print(dataset)
print(dataset['train'].features)

C:\Users\cla5hr\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 16200
    })
    validation: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 5400
    })
    test: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 5400
    })
})
{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']), 'image_id': Value('string')}


In [2]:
# Load processor and model
model_name = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_name)

# Get label mappings
labels = dataset['train'].features['label'].names
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}

print("Labels:", labels)
print("Processor:", processor)

# Preprocess function
def preprocess(batch):
    inputs = processor(images=batch['image'], return_tensors='pt')
    inputs['labels'] = batch['label']
    return inputs

# Apply preprocessing
processed = dataset.with_transform(lambda batch: {
    'pixel_values': processor(images=batch['image'], return_tensors='pt')['pixel_values'],
    'labels': batch['label']
})

print("\nProcessed dataset:", processed)

Labels: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Processor: ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}


Processed dataset: DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 16200
    })
    validation: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 5400
    })
    test: Dataset({
        features: ['image', 'label', 'image_id'],
        num_rows: 5400
    })
})


In [5]:
# Load model
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Accuracy metric
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Fix: preprocess function
def preprocess(examples):
    inputs = processor(images=examples['image'], return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

# Apply correctly
processed = dataset.map(preprocess, batched=True, remove_columns=['image', 'image_id'])
processed.set_format('torch')

print(processed)
print(processed['train'][0])

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 12030.60it/s]
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 5400/5400 [00:10<00:00, 500.30 examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'pixel_values', 'labels'],
        num_rows: 16200
    })
    validation: Dataset({
        features: ['label', 'pixel_values', 'labels'],
        num_rows: 5400
    })
    test: Dataset({
        features: ['label', 'pixel_values', 'labels'],
        num_rows: 5400
    })
})
{'label': tensor(6), 'pixel_values': tensor([[[ 0.5216,  0.5216,  0.5216,  ..., -0.5373, -0.5529, -0.5529],
         [ 0.5216,  0.5216,  0.5216,  ..., -0.5373, -0.5529, -0.5529],
         [ 0.5216,  0.5216,  0.5216,  ..., -0.5294, -0.5451, -0.5451],
         ...,
         [-0.2627, -0.2627, -0.2549,  ...,  0.7412,  0.7412,  0.7412],
         [-0.2314, -0.2314, -0.2235,  ...,  0.7412,  0.7412,  0.7412],
         [-0.2314, -0.2314, -0.2235,  ...,  0.7412,  0.7412,  0.7412]],

        [[ 0.2392,  0.2392,  0.2392,  ..., -0.3804, -0.3961, -0.3961],
         [ 0.2392,  0.2392,  0.2392,  ..., -0.3804, -0.3961, -0.3961],
         [ 0.2392,  0.2392,  0.2392,  .

In [6]:
training_args = TrainingArguments(
    output_dir="./vit-eurosat",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed['train'],
    eval_dataset=processed['validation'],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.148304,0.174694,0.962222
2,0.065780,0.108955,0.974259
3,0.036584,0.063831,0.985741
4,0.016485,0.052039,0.989074
5,0.012343,0.052781,0.989074


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.14s/it]


TrainOutput(global_step=2535, training_loss=0.11447749481163552, metrics={'train_runtime': 2266.3203, 'train_samples_per_second': 35.741, 'train_steps_per_second': 1.119, 'total_flos': 6.277301218234368e+18, 'train_loss': 0.11447749481163552, 'epoch': 5.0})

In [8]:
# Evaluate on test set
results = trainer.predict(processed['test'])
print(results.metrics)

{'test_loss': 0.05738852173089981, 'test_accuracy': 0.9885185185185185, 'test_runtime': 51.9897, 'test_samples_per_second': 103.867, 'test_steps_per_second': 3.251}


In [9]:
trainer.save_model("./vit-eurosat-final")
processor.save_pretrained("./vit-eurosat-final")
print("Model saved!")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.03s/it]

Model saved!


In [11]:
from PIL import Image
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

def predict(image_path):
    image = Image.open(image_path)
    inputs = processor(images=image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    predicted_class = outputs.logits.argmax(-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()
    
    print(f"Predicted: {id2label[predicted_class]}")
    print(f"Confidence: {confidence*100:.2f}%")

# Test
test_image = dataset['test'][0]['image']
test_image.save("test_sample.jpg")
predict("test_sample.jpg")
print(f"Actual: {id2label[dataset['test'][0]['label']]}")

Predicted: Industrial
Confidence: 99.06%
Actual: Industrial
